# Optuna — Búsqueda de hiperparámetros LINE (Cercanías)

**Objetivo:** maximizar una métrica compuesta:

```
score = Spearman(tiempo, distancia_coseno)  −  0.1 × |log(var_FO / var_SO)|
```

- El término **Spearman** mide si el embedding respeta la estructura temporal del grafo.
- La **penalización de varianza** castiga cuando FO y SO tienen varianzas muy distintas entre sí: si una mitad domina la otra en el embedding concatenado, la mitad pequeña no aporta información útil al pipeline MDLF.
  - Penalización = 0 cuando `var_FO == var_SO`
  - Crece logarítmicamente con el desequilibrio (ej. 10× diferencia → penalización ≈ 0.23)

**Estrategia de pruning:** `MedianPruner` con 3 seeds por trial.

```
Hiperparámetros buscados
─────────────────────────────────────────────────────
 dim            int   [8, 64]   step=8
 lr             float [0.001, 0.1]   log-scale
 num_negative   int   [1, 8]
 num_epochs_fo  int   [50, 1500]  step=50
 num_epochs_so  int   [100, 2000] step=100
 init_scale     float [0.01, 1.0] log-scale
 asymmetric     bool  [False, True]
```

In [ ]:
# Instalar Optuna (solo necesario la primera vez)
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "optuna", "optuna-dashboard", "--quiet"],
    check=True
)
import optuna
print(f'Optuna {optuna.__version__} instalado correctamente')

In [ ]:
import sys, pickle, logging, warnings
import numpy as np
import pandas as pd
import networkx as nx
from pathlib import Path
from scipy.stats import spearmanr
from scipy.spatial.distance import cdist

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

# Silenciar logs durante la búsqueda
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from station_line import StationLINE

print('Imports OK')

In [ ]:
CHECK_DIR = Path('../..') / 'cubo_izquierdo' / 'check_transbordo'

with open(CHECK_DIR / 'cercanias_multidigraph_sin_civis.pkl', 'rb') as f:
    G = pickle.load(f)

print(f'Cercanías: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas')

# Comprobación rápida de cuánta señal tiene cada modo
_sym  = StationLINE(dim=4, num_epochs_fo=1, num_epochs_so=1, asymmetric=False)
_dir  = StationLINE(dim=4, num_epochs_fo=1, num_epochs_so=1, asymmetric=True)
_, adj_sym = _sym._build_adjacency(G)
_, adj_dir = _dir._build_adjacency(G)

n_sym = int((np.triu(adj_sym, k=1) > 0).sum())
n_dir = int((adj_dir > 0).sum())
print(f'Pares simétricos (asymmetric=False): {n_sym}')
print(f'Aristas dirigidas (asymmetric=True):  {n_dir}  ({n_dir/n_sym:.1f}x más señal)')

---
## Función objetivo con pruning

Se evalúan **3 seeds** por trial. Tras cada seed se reporta el `score` acumulado (ya con penalización de varianza) y `MedianPruner` cancela el trial si va por debajo de la mediana de los completados en ese paso.

In [ ]:
SEEDS              = [42, 123, 999]
VAR_PENALTY_WEIGHT = 0.1


def spearman_for_emb(emb: pd.DataFrame, model: StationLINE) -> float:
    """Spearman(tiempo, distancia_coseno) sobre pares con arista en el grafo."""
    X = emb.values.astype(float)
    D = cdist(X, X, metric='cosine')

    adj = model.adj_
    with np.errstate(divide='ignore', invalid='ignore'):
        T = np.where(adj > 0, 1.0 / adj, 0.0)

    if model.asymmetric:
        ii, jj = np.where(T > 0)
        t_vals = T[ii, jj]
        d_vals = D[ii, jj]
    else:
        idx    = np.triu_indices(len(model.station_index_), k=1)
        mask   = T[idx] > 0
        t_vals = T[idx][mask]
        d_vals = D[idx][mask]

    r, _ = spearmanr(t_vals, d_vals)
    return float(r) if np.isfinite(r) else 0.0


def var_balance_penalty(emb: pd.DataFrame) -> float:
    """
    |log(var_FO / var_SO)|  — 0 si están equilibradas, crece si divergen.
    Si alguna varianza colapsa a 0 devuelve penalización máxima (10).
    """
    fo_cols = [c for c in emb.columns if c.startswith('fo_')]
    so_cols = [c for c in emb.columns if c.startswith('so_')]
    var_fo  = float(emb[fo_cols].values.var())
    var_so  = float(emb[so_cols].values.var())
    if var_fo <= 0 or var_so <= 0:
        return 10.0
    return abs(np.log(var_fo / var_so))


def objective(trial: optuna.Trial) -> float:
    # --- Espacio de búsqueda ---
    dim        = trial.suggest_int("dim",           8,    64,   step=8)
    lr         = trial.suggest_float("lr",          1e-3, 0.1,  log=True)
    num_neg    = trial.suggest_int("num_negative",  1,    8)
    epochs_fo  = trial.suggest_int("num_epochs_fo", 50,   1500, step=50)
    epochs_so  = trial.suggest_int("num_epochs_so", 100,  2000, step=100)
    init_scale = trial.suggest_float("init_scale",  0.01, 1.0,  log=True)
    asymmetric = trial.suggest_categorical("asymmetric", [False, True])

    scores:   list[float] = []
    r_vals:   list[float] = []
    pen_vals: list[float] = []

    for step, seed in enumerate(SEEDS):
        model = StationLINE(
            dim=dim, lr=lr, num_negative=num_neg,
            num_epochs_fo=epochs_fo, num_epochs_so=epochs_so,
            init_scale=init_scale, asymmetric=asymmetric,
            random_state=seed,
        )
        emb = model.fit_transform(G)

        r   = spearman_for_emb(emb, model)
        pen = var_balance_penalty(emb)

        r_vals.append(r)
        pen_vals.append(pen)
        scores.append(r - VAR_PENALTY_WEIGHT * pen)

        # Reportar score penalizado → MedianPruner compara equivalentes
        trial.report(float(np.mean(scores)), step=step)
        if trial.should_prune():
            raise optuna.TrialPruned()

    # Guardar componentes para análisis posterior
    trial.set_user_attr('mean_spearman', float(np.mean(r_vals)))
    trial.set_user_attr('mean_var_penalty', float(np.mean(pen_vals)))

    return float(np.mean(scores))

---
## Ejecutar la búsqueda

- **100 trials** totales — con pruning, la mayoría acaban antes del seed 3 si son malos.
- `n_startup_trials=10`: los primeros 10 trials se ejecutan completos antes de empezar a podar (el pruner necesita mediana de referencia).
- `n_warmup_steps=1`: no poda en el paso 0 (deja al menos hasta el seed 1 antes de actuar).

In [ ]:
pruner  = MedianPruner(n_startup_trials=10, n_warmup_steps=1, interval_steps=1)
sampler = TPESampler(seed=42)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="LINE_cercanias",
)

study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\nTrials completados : {len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])}')
print(f'Trials podados     : {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}')
print(f'\nMejor Spearman r   : {study.best_value:.4f}')
print(f'Mejores params     :')
for k, v in study.best_params.items():
    print(f'  {k:<18} = {v}')

---
## Análisis de resultados

In [ ]:
import matplotlib.pyplot as plt

complete = [
    t for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
]
top10 = sorted(complete, key=lambda t: t.value, reverse=True)[:10]

rows = [{
    "rank":         i + 1,
    "score":        round(t.value, 4),
    "spearman_r":   round(t.user_attrs.get('mean_spearman', float('nan')), 4),
    "var_penalty":  round(t.user_attrs.get('mean_var_penalty', float('nan')), 4),
    **t.params,
} for i, t in enumerate(top10)]

df_top = pd.DataFrame(rows)
print("Top-10 configuraciones:")
display(df_top)

In [ ]:
# Evolución de la métrica a lo largo de los trials
vals    = [t.value for t in complete]
best_so_far = np.maximum.accumulate(vals)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(range(len(vals)), vals, s=15, alpha=0.6, label='Spearman r (trial)')
ax.plot(range(len(vals)), best_so_far, color='red', linewidth=1.5, label='Mejor hasta ahora')
ax.set_xlabel('Trial completado')
ax.set_ylabel('Spearman r')
ax.set_title('Optuna LINE Cercanías — historia de optimización')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de hiperparámetros (fANOVA — requiere trials suficientes)
try:
    importance = optuna.importance.get_param_importances(study)
    params_imp = list(importance.keys())
    values_imp = list(importance.values())

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.barh(params_imp[::-1], values_imp[::-1], color='steelblue', alpha=0.85)
    ax.set_xlabel('Importancia relativa (fANOVA)')
    ax.set_title('Importancia de hiperparámetros — LINE Cercanías')
    for bar, v in zip(bars, values_imp[::-1]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=8)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Importancia no disponible: {e}')

In [ ]:
# Scatter: valor objetivo vs cada hiperparámetro continuo
param_names = ['lr', 'init_scale', 'num_epochs_fo', 'num_epochs_so', 'dim', 'num_negative']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for ax, pname in zip(axes.flat, param_names):
    xs = [t.params[pname] for t in complete]
    ys = [t.value for t in complete]
    ax.scatter(xs, ys, s=18, alpha=0.6, color='steelblue')
    # Marcar el mejor
    ax.axhline(study.best_value, color='red', linewidth=0.8, linestyle='--', alpha=0.7)
    ax.scatter([study.best_params[pname]], [study.best_value],
               color='red', s=60, zorder=5, label='mejor')
    ax.set_xlabel(pname, fontsize=9)
    ax.set_ylabel('Spearman r', fontsize=8)
    if pname in ('lr', 'init_scale'):
        ax.set_xscale('log')

fig.suptitle('Hiperparámetros vs Spearman r — LINE Cercanías', fontsize=10)
plt.tight_layout()
plt.show()

---
## Entrenamiento final con la mejor configuración

In [ ]:
import time

best = study.best_params
print('Entrenando modelo final con mejores hiperparámetros...')
print(best)

# Promedio de 5 seeds para el modelo de producción
emb_list   = []
model_last = None
t0 = time.time()
for seed in [42, 123, 999, 7, 21]:
    m = StationLINE(
        dim=best['dim'], lr=best['lr'], num_negative=best['num_negative'],
        num_epochs_fo=best['num_epochs_fo'], num_epochs_so=best['num_epochs_so'],
        init_scale=best['init_scale'], asymmetric=best['asymmetric'],
        random_state=seed,
    )
    emb_list.append(m.fit_transform(G))
    model_last = m
print(f'Completado en {time.time()-t0:.1f}s')

# Promedio de embeddings (5 seeds)
emb_best = emb_list[0].copy()
for e in emb_list[1:]:
    emb_best += e
emb_best /= len(emb_list)

# Spearman del modelo final (mismo cálculo que el objetivo Optuna)
X_f = emb_best.values.astype(float)
D_f = cdist(X_f, X_f, metric='cosine')
T_f = np.where(model_last.adj_ > 0, 1.0 / model_last.adj_, 0.0)

if best['asymmetric']:
    ii_f, jj_f = np.where(T_f > 0)
    t_f = T_f[ii_f, jj_f]
    d_f = D_f[ii_f, jj_f]
else:
    idx_f  = np.triu_indices(len(model_last.station_index_), k=1)
    mask_f = T_f[idx_f] > 0
    t_f    = T_f[idx_f][mask_f]
    d_f    = D_f[idx_f][mask_f]

r_final, p_final = spearmanr(t_f, d_f)
print(f'\nSpearman r (modelo final, 5 seeds): {r_final:.4f}  (p={p_final:.2e})')
print(f'Modo: {"dirigido (asymmetric=True)" if best["asymmetric"] else "simétrico (asymmetric=False)"}')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Vecinos más cercanos con la configuración óptima
KEY_HUBS = [
    'SOL', 'MADRID ATOCHA CERCANÍAS', 'MADRID CHAMARTÍN-CLARA CAMPOAMOR',
    'PRÍNCIPE PÍO', 'NUEVOS MINISTERIOS', 'ALCALÁ DE HENARES',
    'GETAFE CENTRO', 'TRES CANTOS',
]
K = 6
station_list = list(emb_best.index)
sim_best     = cosine_similarity(X_f)

def get_line(node, G):
    aps = G.nodes[node].get('access_point_codes', set())
    if isinstance(aps, str): aps = {aps}
    for ap in aps:
        parts = ap.split('_')
        if len(parts) >= 2: return parts[1]
    return 'L?'

print(f'Top-{K} vecinos más cercanos (config óptima)\n')
for hub in KEY_HUBS:
    matches = [s for s in station_list if hub.upper() in s.upper()]
    if not matches:
        print(f'  {hub}: no encontrada\n'); continue
    station = matches[0]
    idx     = station_list.index(station)
    sims    = sim_best[idx]
    top_idx = np.argsort(sims)[::-1][1:K+1]
    print(f'{station}:')
    for rank, j in enumerate(top_idx, 1):
        print(f'  {rank}. {station_list[j]:<45} ({get_line(station_list[j], G)})  sim={sims[j]:.4f}')
    print()

In [ ]:
# Guardar embeddings óptimos y resultados del estudio
out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

emb_best.to_parquet(out_dir / 'line_cercanias_optuna.parquet')
emb_best.round(6).to_csv(out_dir / 'line_cercanias_optuna.csv')

# Guardar tabla de resultados Optuna
df_trials = study.trials_dataframe()
df_trials.to_csv(out_dir / 'optuna_trials_cercanias.csv', index=False)

print(f'Guardado en {out_dir.resolve()}')
print(f'  line_cercanias_optuna.parquet   {emb_best.shape}')
print(f'  optuna_trials_cercanias.csv     {len(df_trials)} trials')

# Resumen final
print(f'\n=== RESUMEN FINAL ===')
print(f'Spearman baseline (params manuales): ~0.436')
print(f'Spearman Optuna best (3 seeds):       {study.best_value:.4f}')
print(f'Spearman modelo final (5 seeds):      {r_final:.4f}')
print(f'Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k:<18} = {v}')